<a href="https://colab.research.google.com/github/SeoYun-Y/thinfiler-credit-scoring/blob/seoyun%2Fperf-analysis/notebooks/track-a/seoyun/seoyun_perf_%ED%99%95%EC%9D%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PERF 타겟변수 및 노출(exposure) 문제 확인

**목적**: Track A(개인CB정보) 데이터의 PERF1~4가 각각 무엇이고, 씬파일러 집단에서
타겟변수로 실제로 쓸 수 있는지 확인한다. 특히
"노출(exposure) 문제"(씬파일러는 신용거래 자체가 없어 PERF가 구조적으로 0에 수렴하는 문제)를
검증하고, 해결 가능한지 확인한다.

**노트북 구성 (목적 순으로 재배열함)**
1. 데이터 로드 및 기본 구조 확인
2. PERF1~4 기초 분포 확인 → 노출 문제 최초 발견
3. 노출 표본을 늘리려는 시도 3가지 → 전부 실패 (실패 원인까지 기록)
4. PERF1/2/4 관계 분석 → 최종 타겟변수 후보 압축
5. PERF3(노출) 추가 진단 → 연도별/인구통계 확인
6. 패널 데이터로 "진짜 노출"을 추적 (씬파일러→일반 전환자) → 노출 조건화 최종 실패 확정
7. 노출편향이 실제로 결과를 왜곡하는지 직접 검증 (씬파일러 vs 일반군 비교) → 최종 해법
8. 결론 정리


## 1. 데이터 로드 및 기본 구조 확인

### 1-1. 드라이브 마운트
공용 계정에서 옮겨받은 데이터가 있는 개인 구글드라이브를 Colab에 연결한다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 1-2. 파일 경로 확인
드라이브 안에 어떤 파일들이 있는지 전체 구조를 먼저 훑어본다.
강사님이 9번(개인CB)·11번(통신카드CB) 데이터를 합쳐서 만들어주신 씬파일러 파일이
`data/개인CB_씬파일러.csv`에 있고, 필터링 전 원본은 `09.개인_CB정보/` 폴더에
연도별(2019-2022)로 나뉘어 있다는 걸 확인한다. 이 노트북의 1-5번은 필터링된 파일을,
6-7번은 원본(연도별) 파일을 사용한다.

In [2]:
import os

# 본인 드라이브 구조에 맞게 경로 확인
base_path = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/'
for root, dirs, files in os.walk(base_path):
    for f in files:
        print(os.path.join(root, f))

/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/093-117_금융 합성데이터_데이터구성상세.xlsx
/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/09.개인_CB정보/202212_개인CB.csv
/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/09.개인_CB정보/202112_개인CB.csv
/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/09.개인_CB정보/202012_개인CB.csv
/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/09.개인_CB정보/201912_개인CB.csv
/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/117.금융_합성데이터/3.개방데이터/1.데이터/1._합성데이터/202103_통신카드CB결합.csv
/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/117.금융_합성데이터/3.개방데이터/1.데이터/1._합성데이터/202106_통신카드CB결합.csv
/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/117.금융_합성데이터/3.개방데이터/1.데이터/1._합성데이터/202109_통신카드CB결합.csv
/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/117.금융_합성데이터/3.개방데이터/1.데이터/1._합성데이터/202112_통신카드CB결합.csv
/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평

### 1-3. 파일 크기 확인
`개인CB_씬파일러.csv`가 1.46GB로 커서, 이후 셀에서는 `usecols`로 필요한 컬럼만
읽어 메모리를 아낀다 (Colab 램 부족 방지).

In [3]:
import os
path_a = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/data/개인CB_씬파일러.csv'
size_mb = os.path.getsize(path_a) / (1024*1024)
print(f"{size_mb:.1f} MB")

1462.9 MB


### 1-4. 컬럼 목록 확인
전체 158개 컬럼이 뭐가 있는지 헤더만 읽어서 확인 (데이터를 실제로 불러오지 않아 메모리 거의 안 씀).

In [4]:
import pandas as pd

path_a = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/data/개인CB_씬파일러.csv'

header = pd.read_csv(path_a, nrows=0)
print(len(header.columns), "개 컬럼")
print(header.columns.tolist())

158 개 컬럼
['STDT', 'ID', 'GENDER', 'AGE_BAND', 'C1Z001373', 'C1M2B4W03', 'C1M2B5W03', 'C1Z001386', 'C1M210000', 'C1M210001', 'C1M210003', 'C1M210006', 'C1811J001', 'C1811J003', 'C1811J006', 'C18210000', 'C18233002', 'C18233003', 'C18233004', 'C18233005', 'C1L120001', 'C1L120003', 'C1L120004', 'C1L120012', 'C1L120017', 'C1L120024', 'C1L120032', 'C1L120041', 'C1L120049', 'C1L120082', 'C1L120084', 'C1L120161', 'C1L120163', 'C1L120167', 'C1L120168', 'C1L120169', 'C1L120170', 'C1L120171', 'C1L120179', 'C1L120196', 'C1L120237', 'C1L120242', 'C1L120250', 'C1L120251', 'C1L120270', 'C1L120272', 'C1L120292', 'L10173000', 'L10210000', 'L10210001', 'L10210002', 'L10210003', 'L10210006', 'L1021000C', 'L1021000F', 'L1021001P', 'L1021003P', 'L1021006P', 'L1021009P', 'L102100CP', 'L1Z001044', 'L90210100', 'L90210200', 'L90210300', 'L10210500', 'L10210800', 'L10210B00', 'L10210M00', 'L10216000', 'L90216100', 'L90216106', 'L90216200', 'L90216300', 'L90216306', 'L90216700', 'L10216800', 'L10216806', 'L102

### 1-5. 상위 몇 줄 미리보기
실제 값이 어떻게 들어있는지 5행만 확인.

In [5]:
preview = pd.read_csv(path_a, nrows=5)
preview

,STDT,ID,GENDER,AGE_BAND,C1Z001373,C1M2B4W03,C1M2B5W03,C1Z001386,C1M210000,C1M210001,...,AP0910001,AP0910002,AS120G001,SCORE,SCORE_6M,PERF1,PERF2,PERF3,PERF4,기준년월
0,201912,83702E5EC3A4D8239C9D3B12F077B15BD09E8BAC4E0097...,2,2,1225,0,0,-7560,0,0,...,9,9,-9,750,759,0,0,0,0,201912
1,201912,0A741C7058923B361931C2CD28158B6DEF991C41711464...,2,6,0,0,0,0,0,0,...,9,9,-9,632,633,0,0,0,0,201912
2,201912,351C2BCF3929F21BA0ADCE54A8C7F0D385F70856C4FE60...,2,7,0,0,0,0,0,0,...,9,9,-9,808,809,0,0,0,0,201912
3,201912,3EE225A1C2DCEFD4EE438FB4B0545900F2CF5FCDF22054...,1,2,-1413,0,0,-3521,0,0,...,9,9,3,745,744,0,0,0,0,201912
4,201912,076C20C1C93707CC1B7E4F2F32727F434DAF0A4F3B5B25...,1,6,-1624,0,0,-19903,0,0,...,9,9,-9,708,699,0,0,0,0,201912


## 2. PERF1~4 기초 분포 확인

**코드북 확인 결과 (093-117_금융 합성데이터_데이터구성상세.xlsx, "9.개인CB 정보" 시트):**

| 필드 | 정의 |
|---|---|
| PERF1 | 신청일로부터 12개월 내 90일 이상 연체경험 여부 |
| PERF2 | 신청일로부터 12개월 내 30일 이상 연체경험 여부 |
| PERF3 | 신청일로부터 3개월 내 신규 대출 발생 여부 (= "노출" 신호) |
| PERF4 | 신청일로부터 6개월 내 60일 이상 연체경험 여부 |

PERF1/2/4는 전부 "연체 여부"이고, PERF3만 성격이 다르다 — 이건 연체가 아니라
"신용거래가 새로 발생했는지"를 나타낸다. 씬파일러는 정의상 카드·대출 이력이 0건이라,
PERF3(신규대출)이 거의 발생하지 않을 것으로 예상되는데 실제로 얼마나 그런지 확인한다.

In [6]:
use_cols = ['STDT', 'ID', 'PERF1', 'PERF2', 'PERF3', 'PERF4']
df_perf = pd.read_csv(path_a, usecols=use_cols)

print(df_perf.shape)
df_perf[['PERF1','PERF2','PERF3','PERF4']].describe()

(3254230, 6)


,PERF1,PERF2,PERF3,PERF4
count,3.254230e+06,3.254230e+06,3.254230e+06,3.254230e+06
mean,9.627469e-04,1.434441e-03,2.304693e-05,8.502779e-04
std,3.101323e-02,3.784684e-02,4.800667e-03,2.914713e-02
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


### 2-1. 각 PERF 컬럼 값별 개수
0/1 외에 다른 값(결측 등)이 섞여 있는지, 각각 몇 명씩인지 확인.

In [7]:
for col in ['PERF1','PERF2','PERF3','PERF4']:
    print(f"\n=== {col} ===")
    print(df_perf[col].value_counts(dropna=False))


=== PERF1 ===
PERF1
0    3251097
1       3133
Name: count, dtype: int64

=== PERF2 ===
PERF2
0    3249562
1       4668
Name: count, dtype: int64

=== PERF3 ===
PERF3
0    3254155
1         75
Name: count, dtype: int64

=== PERF4 ===
PERF4
0    3251463
1       2767
Name: count, dtype: int64


### 2-2. 노출 문제 최초 확인 — PERF3=1인 사람이 얼마나 되는지

**결과: 전체 325만 명 중 PERF3=1은 75명(0.002%)뿐.**
1안(PERF3=1로 노출 필터링 후 PERF4를 타겟으로)을 그대로 쓰면
표본 75개로 학습해야 한다는 뜻 — 사실상 불가능한 크기다. 이게 이 노트북에서
풀어야 할 핵심 문제가 됐다.

In [8]:
exposed = df_perf[df_perf['PERF3'] == 1]
print(f"PERF3=1(노출) 인원: {len(exposed):,} / 전체 {len(df_perf):,} ({len(exposed)/len(df_perf)*100:.2f}%)")
print(exposed['PERF4'].value_counts(dropna=False))

PERF3=1(노출) 인원: 75 / 전체 3,254,230 (0.00%)
PERF4
0    75
Name: count, dtype: int64


## 3. 노출 표본을 늘리려는 시도 — 전부 실패 (원인 기록용으로 남김)

PERF3=75명이라는 표본을 늘려보려고 3가지를 시도했다. 결과적으로 **셋 다 실패**했는데,
왜 실패했는지가 다음 단계 판단에 중요한 정보라 지우지 않고 남겨둔다.

### 3-1. 시도① 카드·대출 보유 조건 완화

씬파일러 정의(카드·대출 관련 5개 필드 전부 0)를 "1건까지는 허용"으로 완화하면
노출(PERF3) 표본이 늘어날 거라 기대하고 시도.

**⚠️ 이 시도는 무효였다**: 지금 쓰는 `개인CB_씬파일러.csv` 파일 자체가 강사님이
**이미 원래 정의(5개 필드 전부 0)로 필터링해서 만들어주신 파일**이다. 즉 이 파일
안에는 "카드 1개 보유자" 같은 사람이 애초에 존재하지 않아서, `<=1` 같은 완화 조건을
걸어도 전부 그대로 통과되어 버린다 (원래 값이 다 0이라서). 그래서 조건2·3·4 결과가
전부 원본(조건 없음)과 완전히 똑같이 나왔다 — 완화가 실제로는 전혀 작동하지 않은 것.

In [9]:
use_cols_full = ['STDT', 'ID', 'C1M210000', 'C18233003', 'C18233004', 'C18233005',
                  'L10220000', 'PERF1','PERF2','PERF3','PERF4']
df_full = pd.read_csv(path_a, usecols=use_cols_full)

# 조건2: 카드 관련 4개 필드는 0, 대출기관수(L10220000)는 1 이하
cond2 = (df_full['C1M210000']==0) & (df_full['C18233003']==0) & \
        (df_full['C18233004']==0) & (df_full['C18233005']==0) & \
        (df_full['L10220000'] <= 1)

df_cond2 = df_full[cond2]
print(f"조건2 인원: {len(df_cond2):,}")
for col in ['PERF1','PERF2','PERF3','PERF4']:
    print(f"{col}: {(df_cond2[col]==1).sum():,}명 ({(df_cond2[col]==1).mean()*100:.3f}%)")

조건2 인원: 3,254,230
PERF1: 3,133명 (0.096%)
PERF2: 4,668명 (0.143%)
PERF3: 75명 (0.002%)
PERF4: 2,767명 (0.085%)


조건 3 — 카드도 완화 (카드 1개까지, 대출은 원래 0)

In [10]:
cond3 = (df_full['C1M210000']<=1) & (df_full['C18233003']<=1) & \
        (df_full['C18233004']<=1) & (df_full['C18233005']<=1) & \
        (df_full['L10220000'] == 0)

df_cond3 = df_full[cond3]
print(f"조건3 인원: {len(df_cond3):,}")
for col in ['PERF1','PERF2','PERF3','PERF4']:
    print(f"{col}: {(df_cond3[col]==1).sum():,}명 ({(df_cond3[col]==1).mean()*100:.3f}%)")

조건3 인원: 3,254,230
PERF1: 3,133명 (0.096%)
PERF2: 4,668명 (0.143%)
PERF3: 75명 (0.002%)
PERF4: 2,767명 (0.085%)


조건4 — 카드·대출 둘 다 1건까지 완화 (가장 넓은 완화). 이것도 결과 동일 → 완화 자체가 이 파일에서는 무의미함을 최종 확인.

In [11]:
cond4 = (df_full['C1M210000']<=1) & (df_full['C18233003']<=1) & \
        (df_full['C18233004']<=1) & (df_full['C18233005']<=1) & \
        (df_full['L10220000'] <= 1)

df_cond4 = df_full[cond4]
print(f"조건4 인원: {len(df_cond4):,}")
for col in ['PERF1','PERF2','PERF3','PERF4']:
    print(f"{col}: {(df_cond4[col]==1).sum():,}명 ({(df_cond4[col]==1).mean()*100:.3f}%)")

조건4 인원: 3,254,230
PERF1: 3,133명 (0.096%)
PERF2: 4,668명 (0.143%)
PERF3: 75명 (0.002%)
PERF4: 2,767명 (0.085%)


### 3-2. 시도② SCORE(신용점수) 기준으로 다르게 걸러보기

"SCORE=0(K-Score 적용배제대상)인 사람 때문에 노출이 안 잡히는 거 아닐까?" 하는
가설로, SCORE≠0(평가 가능 대상)인 사람만 따로 봤다.

**결과: 거의 변화 없음.** 씬파일러는 이미 대부분 SCORE≠0(평가 가능 대상)이었다.
즉 SCORE는 노출 문제와 무관 — 노출이 적은 진짜 이유는 "씬파일러가 원래
신규 대출 이벤트 자체가 거의 없는 집단"이기 때문이라는 게 다시 확인됨.

In [12]:
use_cols_score = ['ID', 'SCORE', 'PERF1','PERF2','PERF3','PERF4']
df_score = pd.read_csv(path_a, usecols=use_cols_score)

# SCORE가 0이 아닌(=평가 가능) 사람 중 PERF 분포
scored = df_score[df_score['SCORE'] != 0]
print(f"SCORE≠0 인원: {len(scored):,}")
for col in ['PERF1','PERF2','PERF3','PERF4']:
    print(f"{col}: {(scored[col]==1).sum():,}명")

SCORE≠0 인원: 3,253,521
PERF1: 3,132명
PERF2: 4,667명
PERF3: 75명
PERF4: 2,767명


### 3-3. 시도③ 대체 노출 신호 탐색 ("신규개설" 계열 필드)

PERF3("3개월 내 신규대출")보다 관찰 기간이 긴 필드들
(`C1M210001/003/006`=신용카드 1/3/6개월내 신규개설, `L10210001/002/003/006`=대출 1/2/3/6개월내
신규개설)을 대신 써서 표본을 늘릴 수 있는지 확인. 이 필드들은 씬파일러 정의에
쓰이지 않은 필드라 값이 0이 아닐 여지가 있다.

In [13]:
use_cols_alt = ['ID', 'C1M210001', 'C1M210003', 'C1M210006',
                'L10210001', 'L10210002', 'L10210003', 'L10210006',
                'PERF1','PERF2','PERF3','PERF4']
df_alt = pd.read_csv(path_a, usecols=use_cols_alt)

for col in ['C1M210001','C1M210003','C1M210006','L10210001','L10210002','L10210003','L10210006']:
    n = (df_alt[col] > 0).sum()
    print(f"{col} > 0 인 사람: {n:,}명")

C1M210001 > 0 인 사람: 2명
C1M210003 > 0 인 사람: 1,440명
C1M210006 > 0 인 사람: 7,339명
L10210001 > 0 인 사람: 1명
L10210002 > 0 인 사람: 66명
L10210003 > 0 인 사람: 602명
L10210006 > 0 인 사람: 11,052명


표본은 확실히 늘었다 (특히 L10210006=6개월내 대출신규개설: 11,052명). 이 대체 신호를
"노출"로 보고, 그 안에서 실제 연체(PERF1/2/4)가 얼마나 나오는지 확인.

**결과: 오히려 전체 평균보다 낮게 나옴** (PERF1 0.081% vs 전체 0.096% 등).
"신규 대출이 생긴 사람일수록 연체 위험도 높을 것"이라는 기대와 반대로 나왔다.
원인으로 추정되는 것: PERF는 "신청일 이후 미래" 시점을 보는 라벨인데, 이 필드는
"기준일 기준 과거 6개월"을 보는 값이라 시간 기준이 어긋나 있을 가능성 — 이 대체
신호로는 노출을 대신할 수 없다는 결론.

In [14]:
# L10210006을 노출 신호로 써서, 그 안에서 PERF1/2/4 분포 확인
exposed_alt = df_alt[df_alt['L10210006'] > 0]
print(f"L10210006>0 (노출) 인원: {len(exposed_alt):,}")
for col in ['PERF1','PERF2','PERF4']:
    print(f"{col}: {(exposed_alt[col]==1).sum():,}명 ({(exposed_alt[col]==1).mean()*100:.3f}%)")

L10210006>0 (노출) 인원: 11,052
PERF1: 9명 (0.081%)
PERF2: 14명 (0.127%)
PERF4: 5명 (0.045%)


### 3장 소결

세 가지 시도(조건완화 / SCORE필터 / 대체신호) 전부 PERF3(노출) 표본을 의미 있게
늘리지 못했다. **씬파일러 안에서 노출 표본을 억지로 늘리는 방향은 막다른 길**이라는
결론. 다음 두 갈래로 방향을 바꾼다:
- 4장: PERF3(노출)은 접어두고, 연체 라벨인 PERF1/2/4 중 무엇을 최종 타겟으로 쓸지 결정
- 6장: 노출을 "억지로 늘리기"가 아니라 "실제로 발생한 시점을 시간축(연도별)에서 추적"하는 방식으로 재도전

## 4. PERF1/2/4 관계 분석 — 최종 타겟변수 후보 압축

### 4-1. 대안변수 하나(생활안정성)와 PERF1의 관계 먼저 확인

노출 문제와 별개로, "정의를 안 바꿔도 지금 씬파일러 데이터 안에서 대안변수가
연체(PERF1)를 구분하는 힘이 있는지" 확인. AL012G005(3년내 자택주소 이력건수,
구간화)와 PERF1을 교차표로 봄.

In [15]:
use_cols_iv = ['ID', 'AL012G005', 'AL012G011', 'AL012G019', 'U81203010',
               'PERF1','PERF2','PERF4']
df_iv = pd.read_csv(path_a, usecols=use_cols_iv)

pd.crosstab(df_iv['AL012G005'], df_iv['PERF1'], normalize='index')

PERF1,0,1
AL012G005,,
-9,1.000000,0.000000
1,0.999437,0.000563
2,0.999061,0.000939
3,0.997435,0.002565
4,0.996126,0.003874
5,0.988264,0.011736
6,0.990757,0.009243
7,0.990344,0.009656


이사 이력이 많을수록(구간이 올라갈수록) PERF1=1 비율이 대체로 올라가는 패턴이
보인다 (구간 6,7에서 약간 흔들리지만). 이걸 IV(Information Value)로 정량화해서
"이 변수가 얼마나 강하게 연체를 구분하는지" 숫자로 확인한다.

**IV 해석 기준**: <0.02 거의 없음 / 0.02~0.1 약함 / 0.1~0.3 중간 / >0.3 강함

In [16]:
import numpy as np

def calc_iv(df, feature, target):
    grouped = df.groupby(feature)[target].agg(['count','sum'])
    grouped.columns = ['total','bad']
    grouped['good'] = grouped['total'] - grouped['bad']

    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()

    grouped['bad_rate'] = grouped['bad'] / total_bad
    grouped['good_rate'] = grouped['good'] / total_good

    # 0 방지용 스무딩
    grouped['bad_rate'] = grouped['bad_rate'].replace(0, 0.0001)
    grouped['good_rate'] = grouped['good_rate'].replace(0, 0.0001)

    grouped['woe'] = np.log(grouped['good_rate'] / grouped['bad_rate'])
    grouped['iv'] = (grouped['good_rate'] - grouped['bad_rate']) * grouped['woe']

    return grouped, grouped['iv'].sum()

result, iv_value = calc_iv(df_iv, 'AL012G005', 'PERF1')
print(result)
print(f"\nIV = {iv_value:.4f}")

             total   bad     good  bad_rate  good_rate       woe        iv
AL012G005                                                                 
-9              11     0       11  0.000100   0.000003 -3.386267  0.000327
 1         1527001   859  1526142  0.274178   0.469424  0.537728  0.104989
 2         1449494  1361  1448133  0.434408   0.445429  0.025054  0.000276
 3          218336   560   217776  0.178742   0.066985 -0.981471  0.109686
 4           42077   163    41914  0.052027   0.012892 -1.395132  0.054598
 5           11759   138    11621  0.044047   0.003574 -2.511441  0.101645
 6            3895    36     3859  0.011491   0.001187 -2.270112  0.023390
 7            1657    16     1641  0.005107   0.000505 -2.314284  0.010651

IV = 0.4056


**IV = 0.4056 → 강한 예측력.** 다만 위 표의 `bad_rate` 컬럼명은 "전체 연체자 중 이
구간이 차지하는 비중"이라 헷갈리기 쉽다. 실제 "구간별 연체율"(이 구간 안에서 연체한
사람의 비율)은 아래처럼 따로 계산해서 확인해야 한다.

In [17]:
result['actual_bad_rate'] = result['bad'] / result['total'] * 100
print(result[['total','bad','actual_bad_rate']])

             total   bad  actual_bad_rate
AL012G005                                
-9              11     0         0.000000
 1         1527001   859         0.056254
 2         1449494  1361         0.093895
 3          218336   560         0.256485
 4           42077   163         0.387385
 5           11759   138         1.173569
 6            3895    36         0.924262
 7            1657    16         0.965600


실제 연체율도 이사 이력 구간이 올라갈수록 계속 올라가는 패턴(0.06% → 1.17%)으로
확인됨 — IV 값이 유효하다는 게 재확인됐다.

### 4-2. PERF1/2/4를 하나로 합친 "통합 타겟" 시도

노출(PERF3) 표본이 부족하니, PERF1/2/4 중 하나라도 1이면 1로 보는 통합 타겟을
만들어서 표본이 얼마나 느는지 확인.

In [18]:
use_cols_p3 = ['STDT', 'ID', 'PERF1','PERF2','PERF3','PERF4']
df_p3 = pd.read_csv(path_a, usecols=use_cols_p3)

df_p3['PERF_any'] = ((df_p3['PERF1']==1) | (df_p3['PERF2']==1) | (df_p3['PERF4']==1)).astype(int)
print(df_p3['PERF_any'].value_counts())
print(f"비율: {df_p3['PERF_any'].mean()*100:.3f}%")

PERF_any
0    3249551
1       4679
Name: count, dtype: int64
비율: 0.144%


통합해도 4,679명으로, PERF2 단독(4,668명)과 거의 같다. 즉 PERF1과 PERF4는
PERF2에 대부분 포함되는 부분집합이라는 뜻 — 아래 교차표로 정확한 포함관계를 확인.

In [19]:
print(pd.crosstab(df_p3['PERF1'], df_p3['PERF2']))
print(pd.crosstab(df_p3['PERF1'], df_p3['PERF4']))

PERF2        0     1
PERF1               
0      3249555  1542
1            7  3126
PERF4        0     1
PERF1               
0      3251074    23
1          389  2744


**결과 해석**
- PERF1=1인 사람의 99.8%가 PERF2=1이기도 함 → 당연한 포함관계 (90일 연체면 그 전에 이미 30일도 지났으므로 PERF2 조건도 만족)
- PERF1=1인 사람의 87.6%만 PERF4=1 → 관찰기간 차이(PERF1은 12개월, PERF4는 6개월) 때문에 발생하는 자연스러운 불일치

**결론**: `PERF2(30일↑연체, 12개월) ⊃ PERF1(90일↑연체, 12개월) ⊃ 거의 PERF4(60일↑연체, 6개월)`
구조. **PERF2가 표본이 제일 많고(4,668명) 나머지를 대부분 포함하는 상위 기준**이라
최종 타겟변수 후보 1순위.

## 5. PERF3(노출) 추가 진단 — 75명이 어떤 사람들인지

### 5-1. 연도별 분포
PERF3=1인 75명이 특정 연도에 몰려있는지 확인.

In [20]:
exposed_p3 = df_p3[df_p3['PERF3']==1]
print(exposed_p3['STDT'].value_counts())

STDT
202212    53
202112    17
202012     3
201912     2
Name: count, dtype: int64


**결과: 2019년 2명 → 2020년 3명 → 2021년 17명 → 2022년 53명으로 계속 증가.**
씬파일러가 시간이 지나며 점점 신용시스템에 편입되는 자연스러운 흐름으로 보이지만,
2022년(가장 표본이 많은 해)도 53명뿐이라 여전히 모델링하기엔 턱없이 부족하다.

### 5-2. 인구통계 확인
PERF3=1인 75명이 특정 연령대·성별에 쏠려있는지 확인 (특이 집단인지, 아니면
씬파일러 전체 분포를 그대로 따르는지).

In [21]:
use_cols_demo = ['ID', 'GENDER', 'AGE_BAND', 'PERF3']
df_demo = pd.read_csv(path_a, usecols=use_cols_demo)

exposed_demo = df_demo[df_demo['PERF3']==1]
print(exposed_demo['AGE_BAND'].value_counts())
print(exposed_demo['GENDER'].value_counts())

AGE_BAND
2    43
3    16
4    12
5     4
Name: count, dtype: int64
GENDER
1    52
2    23
Name: count, dtype: int64


57%가 20대(AGE_BAND=2)로 나왔는데, 이는 씬파일러 모집단 자체가 원래 20대
비중이 높은 것과 일치 — 노출이 특정 집단에 쏠려서 생긴 특이현상은 아닌 것으로 보임.

### 5-3. 패널(다중연도) 표본 규모 확인
이후 6장에서 "작년엔 씬파일러였는데 올해는 아닌 사람(탈출자)"을 찾으려면, 같은
사람이 여러 연도에 걸쳐 나타나야 한다. 이게 실제로 얼마나 되는지 먼저 확인.

In [22]:
id_counts = df_p3['ID'].value_counts()
print(f"2개 연도 이상 등장한 ID 수: {(id_counts>=2).sum():,}")

2개 연도 이상 등장한 ID 수: 839,183


83만 명(전체의 약 26%)이 2개 연도 이상 나타남 — 패널 추적에 쓸 표본은 충분하다는 것을 확인.

## 6. 패널 데이터로 "진짜 노출"을 추적 — 씬파일러→일반 전환자(탈출자) 찾기

**아이디어**: 지금까지는 한 시점(스냅샷) 안에서 노출 신호를 찾으려 했다. 대신
"작년엔 씬파일러(카드·대출 0건)였는데, 올해는 카드나 대출이 생긴 사람"을 찾으면
이게 곧 "노출이 실제로 발생한 순간"이 된다.

**주의**: 지금 쓰던 `개인CB_씬파일러.csv`는 이미 씬파일러로 필터링된 파일이라
"씬파일러였다가 탈출한 사람"은 애초에 이 파일에 없다 (탈출하면 필터 조건을 벗어나
빠지기 때문). 그래서 여기서부터는 **원본(필터 전) 연도별 파일**
(`09.개인_CB정보/` 폴더)을 사용한다.

### 6-1. 필요한 연도(2021, 2022) 파일에서 씬파일러 여부 플래그 생성

313만 건씩인 원본 파일 4개를 한꺼번에 메모리에 올리면 램 부족이 나서, 필요한
컬럼만 읽고 쓰자마자 gc.collect()로 메모리를 정리하는 함수로 처리한다.

In [23]:
import gc

base = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/09.개인_CB정보'

use_cols_panel = ['STDT', 'ID', 'C1M210000', 'C18233003', 'C18233004', 'C18233005',
                   'L10220000', 'PERF1','PERF2','PERF3','PERF4']

def load_thin_flag(path):
    df_tmp = pd.read_csv(path, usecols=use_cols_panel)
    df_tmp['is_thin'] = ((df_tmp['C1M210000']==0) & (df_tmp['C18233003']==0) &
                          (df_tmp['C18233004']==0) & (df_tmp['C18233005']==0) &
                          (df_tmp['L10220000']==0)).astype(int)
    result = df_tmp[['ID','is_thin','PERF1','PERF2','PERF3','PERF4']].copy()
    del df_tmp
    gc.collect()
    return result

# 2021, 2022 두 연도만 (2019~2022 전체를 보고 싶으면 같은 방식으로 반복하면 됨)
prev = load_thin_flag(f'{base}/202112_개인CB.csv').rename(columns={'is_thin':'is_thin_prev'})
prev = prev[['ID','is_thin_prev']]
gc.collect()

curr = load_thin_flag(f'{base}/202212_개인CB.csv')

### 6-2. 2021→2022 전환자(탈출자) 찾기

작년(2021)엔 씬파일러(is_thin_prev=1)였는데 올해(2022)는 아닌(is_thin=0) 사람만 골라서,
그 안에서 PERF1~4 분포를 확인한다.

In [24]:
merged = curr.merge(prev, on='ID', how='inner')  # 두 연도 모두 존재하는 사람만
del curr, prev
gc.collect()

print(f"2021,2022 둘 다 있는 사람: {len(merged):,}")

graduated = merged[(merged['is_thin_prev']==1) & (merged['is_thin']==0)].copy()
print(f"탈출자(노출 발생) 인원: {len(graduated):,}")

for col in ['PERF1','PERF2','PERF3','PERF4']:
    print(f"{col}: {(graduated[col]==1).sum():,}명 ({(graduated[col]==1).mean()*100:.3f}%)")

del merged
gc.collect()

2021,2022 둘 다 있는 사람: 3,129,036
탈출자(노출 발생) 인원: 51,917
PERF1: 13명 (0.025%)
PERF2: 46명 (0.089%)
PERF3: 2,348명 (4.523%)
PERF4: 9명 (0.017%)


0

**핵심 발견: 탈출자 51,917명 중 PERF3=1이 4.523%(2,348명).**
전체 씬파일러 기준 PERF3 비율(0.002%)과 비교하면 2,000배 넘게 뛰었다. 이건
PERF3이 원래 이상하게 적은 값이었던 게 아니라, **"신용거래를 실제로 시작한
사람들" 안에서 보면 정상적인 비율(몇 %)로 나온다**는 뜻 — 노출을 시간축으로
추적하는 이 접근이 방향은 맞았다는 신호.

### 6-3. 이 탈출자 그룹 안에서 실제 연체(PERF4)까지 확인

1안(PERF3=1 필터 → PERF4 타겟)을 이제 이 탈출자 정의로 살릴 수
있는지, 마지막 단계까지 표본이 남는지 확인한다.

In [25]:
exposed_final = graduated[graduated['PERF3']==1]
print(f"최종 노출집단(탈출자 + PERF3=1): {len(exposed_final):,}")
for col in ['PERF1','PERF2','PERF4']:
    print(f"{col}: {(exposed_final[col]==1).sum():,}명 ({(exposed_final[col]==1).mean()*100:.3f}%)")

최종 노출집단(탈출자 + PERF3=1): 2,348
PERF1: 1명 (0.043%)
PERF2: 8명 (0.341%)
PERF4: 1명 (0.043%)


**결과: 2,348명 중 PERF4=1은 단 1명.** 조건을 세 겹(전체→탈출자→PERF3=1)으로
좁힐 때마다 표본이 기하급수적으로 줄어서, 최종 연체 예측 단계에서 다시 학습
불가능한 크기가 됐다.

### 6장 소결

**PERF3 기반 노출 조건화(1안)는 어떤 방식으로 접근해도 이 데이터 안에서는
표본 부족으로 실행 불가능**하다는 게 최종 확인됐다. 다만 왜 노출 표본이
원래 이렇게 적었는지는 명확히 설명됐다 (씬파일러 대다수가 아직 신용거래를
시작하지 않았을 뿐, 데이터 이상은 아님). 다음 장에서는 관점을 바꿔서,
"노출 조건화 없이 PERF2를 바로 타겟으로 써도 되는지"를 다른 방식으로 검증한다.

### 6-4. 정의 완화로 재시도 (원본 데이터 기반)

패널 방식(탈출자 추적)과는 별개로, 원본 데이터가 있으니 어제 미결이었던
"B안(씬파일러 정의를 0건 → 1건 이하로 완화)"을 직접 원본에서 테스트해본다.
(1장~3장에서 썼던 `개인CB_씬파일러.csv`는 이미 필터링된 파일이라 완화 시도가
무효했었는데, 여기서는 진짜 원본이라 완화가 실제로 작동한다.)

2022년 한 해만 먼저 테스트한다 (메모리 고려).

In [26]:
path_2022_raw = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/09.개인_CB정보/202212_개인CB.csv'

use_cols_relax = ['ID', 'C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000',
                   'PERF1','PERF2','PERF3','PERF4']
df_raw22 = pd.read_csv(path_2022_raw, usecols=use_cols_relax)

# 원래 정의 (5개 필드 전부 0)
cond_orig = ((df_raw22['C1M210000']==0) & (df_raw22['C18233003']==0) &
             (df_raw22['C18233004']==0) & (df_raw22['C18233005']==0) &
             (df_raw22['L10220000']==0))

# 완화 정의 (5개 필드 전부 1건 이하)
cond_relaxed = ((df_raw22['C1M210000']<=1) & (df_raw22['C18233003']<=1) &
                (df_raw22['C18233004']<=1) & (df_raw22['C18233005']<=1) &
                (df_raw22['L10220000']<=1))

df_orig = df_raw22[cond_orig]
df_relaxed = df_raw22[cond_relaxed]

print(f"원래 정의(0건): {len(df_orig):,}명")
print(f"완화 정의(1건 이하): {len(df_relaxed):,}명")

for col in ['PERF1','PERF2','PERF3','PERF4']:
    n_orig = (df_orig[col]==1).sum()
    n_relaxed = (df_relaxed[col]==1).sum()
    print(f"{col} — 원래: {n_orig:,}명 / 완화: {n_relaxed:,}명")

원래 정의(0건): 794,773명
완화 정의(1건 이하): 1,496,539명
PERF1 — 원래: 337명 / 완화: 3,583명
PERF2 — 원래: 414명 / 완화: 4,085명
PERF3 — 원래: 53명 / 완화: 2,191명
PERF4 — 원래: 315명 / 완화: 3,432명


**결과 (2022년 기준)**

| | 원래 정의(0건) | 완화 정의(1건 이하) | 배율 |
|---|---|---|---|
| 인원 | 794,773명 | 1,496,539명 | 1.9배 |
| PERF1 | 337명 | 3,583명 | 10.6배 |
| PERF2 | 414명 | 4,085명 | 9.9배 |
| **PERF3** | **53명** | **2,191명** | **41.3배** |
| PERF4 | 315명 | 3,432명 | 10.9배 |

인원은 1.9배만 늘었는데 PERF3(노출)은 41배나 늘었다 — 완화가 "이미 신용거래를
시작한 사람들"을 정확히 골라내는 효과를 낸다는 뜻이다 (1~3장에서 필터링된 파일로
시도했을 때는 완화 자체가 작동하지 않아 이 효과를 볼 수 없었다). PERF3=2,191명이면
드디어 모델링 가능한 크기 — 이 안에서 실제 연체(PERF4)가 얼마나 나오는지 마지막으로
확인한다.

In [27]:
exposed_relaxed = df_relaxed[df_relaxed['PERF3']==1]
print(f"완화정의 노출집단: {len(exposed_relaxed):,}명")
for col in ['PERF1','PERF2','PERF4']:
    print(f"{col}: {(exposed_relaxed[col]==1).sum():,}명 ({(exposed_relaxed[col]==1).mean()*100:.3f}%)")

완화정의 노출집단: 2,191명
PERF1: 0명 (0.000%)
PERF2: 1명 (0.046%)
PERF4: 0명 (0.000%)


**결과: 완화정의 노출집단 2,191명 중 PERF1=0명, PERF2=1명, PERF4=0명.**

노출(PERF3) 표본은 41배 늘었지만, 정작 최종 타겟인 PERF4는 0명 — 6장 앞부분의
패널(탈출자) 방식 결과(2,348명 중 PERF4=1명)와 사실상 같은 결말이다.

**두 가지 서로 다른 방법(패널 기반 탈출자 추적 / 정의 완화)으로 각각 노출 표본
확보에는 성공했지만, 둘 다 "노출 → 그 직후 단기간 내 연체"로 이어지는 마지막
단계에서 표본이 0~1명으로 사라졌다.** 이게 두 번 반복됐다는 건 우리 방법이
부족해서가 아니라, **"신규 대출 발생 직후 짧은 기간(6개월) 안에 연체까지 가는
사건" 자체가 원래 희귀한 현상**이라는 걸 시사한다 — 씬파일러가 아닌 일반 고객
중에서도 대출 받자마자 몇 달 안에 연체하는 경우는 흔치 않기 때문.

## 7. 노출편향이 실제로 결과를 왜곡하는지 직접 검증 (최종 해법)

**아이디어 전환**: "노출 표본을 늘리는 것"은 포기한다. 대신 "PERF2를 노출 조건화
없이 바로 타겟으로 쓰면, 걱정했던 왜곡(모델이 연체가 아니라 신용거래 개시 여부를
배우는 것)이 실제로 일어나는지"를 직접 검증한다.

**검증 논리**: 이미 신용거래를 하고 있는 일반 고객군은 노출 문제가 아예 없다
(PERF가 순수하게 "진짜 연체했는지"만 나타냄). 씬파일러에서 나온 "대안변수-연체"
관계가 일반 고객군에서도 같은 방향으로 나온다면, 이 관계는 노출편향이 아니라
진짜 위험신호라고 볼 수 있다.

### 7-1. 일반 고객군(이미 노출된 집단)에서 같은 변수-PERF1 관계 확인

2022년 원본 데이터에서 씬파일러가 아닌 사람들(카드·대출 있는 일반 고객)만 뽑아서,
AL012G005(생활안정성)와 PERF1의 관계를 씬파일러와 같은 방식으로 계산한다.

In [28]:
path_2022 = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터/09.개인_CB정보/202212_개인CB.csv'

use_cols_gen = ['ID', 'C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000',
                'AL012G005', 'PERF1']
df_2022 = pd.read_csv(path_2022, usecols=use_cols_gen)

is_thin = ((df_2022['C1M210000']==0) & (df_2022['C18233003']==0) &
           (df_2022['C18233004']==0) & (df_2022['C18233005']==0) &
           (df_2022['L10220000']==0))

df_general = df_2022[~is_thin]  # 일반 고객군 (노출 문제 없음)
print(f"일반 고객군: {len(df_general):,}명")

result_general, iv_general = calc_iv(df_general, 'AL012G005', 'PERF1')
print(result_general[['total','bad']].assign(
    actual_bad_rate=lambda x: x['bad']/x['total']*100))
print(f"\n일반군 IV = {iv_general:.4f}")

일반 고객군: 2,334,263명
             total   bad  actual_bad_rate
AL012G005                                
-9              33     0         0.000000
 1          515551  1042         0.202114
 2         1444766  7021         0.485961
 3          241033  1665         0.690777
 4           49139   309         0.628828
 5           57681   661         1.145958
 6           18197   212         1.165027
 7            7863    91         1.157319

일반군 IV = 0.1758


**결과: 일반군에서도 이사 이력이 많을수록 연체율이 올라가는 같은 방향의 패턴이
재현됨** (0.20% → 1.16%). IV도 0.1758로 "중간~강한 예측력" 구간. 씬파일러 IV(0.4056)보다는
낮지만, 방향이 일치한다는 게 핵심.

### 7-2. IV 격차가 "진짜 차이"인지 "불균형 때문 착시"인지 검증

씬파일러 쪽 IV가 일반군보다 2배 넘게 높은 게, 씬파일러의 극단적 불균형(연체 0.1%
미만)이 만든 통계적 착시일 수도 있다. 이를 확인하기 위해 일반군 데이터를
씬파일러 수준(1:1000)의 불균형으로 인위적으로 맞춰서 IV를 다시 계산한다.

**⚠️ 첫 시도는 실패** — good(정상) 쪽을 늘려서 비율을 맞추려 했는데, 필요한
표본 수(1,100만 명)가 실제 일반군 전체 인구(233만 명)보다 많아서 에러 발생.
good 대신 **bad(연체자) 쪽을 줄이는 방식**으로 다시 시도해야 한다.

In [29]:
bad = df_general[df_general['PERF1']==1]
good_pool = df_general[df_general['PERF1']==0]

target_ratio = 1000  # 씬파일러 수준 불균형 재현 목표

# 첫 시도 (실패 예상): good을 bad*1000만큼 늘리려 하면 모집단 부족으로 에러 남
# n_good = len(bad) * target_ratio  # 이 값이 good_pool 전체보다 큼 → ValueError

# 올바른 방식: bad(연체자)를 줄여서 비율을 맞춘다
np.random.seed(42)
n_bad = max(1, len(good_pool) // target_ratio)
n_bad = min(n_bad, len(bad))

bad_sampled = bad.sample(n=n_bad, random_state=42)
df_general_resampled = pd.concat([bad_sampled, good_pool])

print(f"bad: {len(bad_sampled):,}, good: {len(good_pool):,}, 불균형비: 1:{len(good_pool)/len(bad_sampled):.0f}")

_, iv_resampled = calc_iv(df_general_resampled, 'AL012G005', 'PERF1')
print(f"일반군(불균형 맞춤 후) IV = {iv_resampled:.4f}")

bad: 2,323, good: 2,323,262, 불균형비: 1:1000
일반군(불균형 맞춤 후) IV = 0.1528


**결과: 불균형을 씬파일러 수준(1:1000)으로 맞춰도 일반군 IV는 0.1528로,
여전히 씬파일러(0.4056)보다 훨씬 낮다.** 즉 IV 격차는 불균형 때문에 생긴
착시가 아니라 **진짜 차이**다. → 대안변수(생활안정성)가 씬파일러 집단에서
유독 더 강하게 연체를 예측한다는 뜻이고, 이는 "전통 금융이력이 없을수록
비금융 대안정보의 가치가 커진다"는 프로젝트 핵심 가설과 부합하는 결과다.

### 7-3. 다른 대안변수들도 같은 패턴이 재현되는지 반복 검증

AL012G005 하나만으로는 우연일 수 있으니, 나머지 대안변수 3개(AL012G011, AL012G019,
U81203010)에 대해서도 씬파일러군 vs 일반군 IV를 비교한다.

In [30]:
alt_vars = ['AL012G011', 'AL012G019', 'U81203010']

for var in alt_vars:
    use_cols_v = ['ID', var, 'PERF1', 'C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000']
    df_v = pd.read_csv(path_2022, usecols=use_cols_v)
    is_thin_v = ((df_v['C1M210000']==0) & (df_v['C18233003']==0) &
                 (df_v['C18233004']==0) & (df_v['C18233005']==0) & (df_v['L10220000']==0))

    _, iv_thin = calc_iv(df_v[is_thin_v], var, 'PERF1')
    _, iv_gen = calc_iv(df_v[~is_thin_v], var, 'PERF1')
    print(f"{var}: 씬파일러 IV={iv_thin:.4f}, 일반군 IV={iv_gen:.4f}")

AL012G011: 씬파일러 IV=0.1722, 일반군 IV=0.0320
AL012G019: 씬파일러 IV=1.3319, 일반군 IV=0.6542
U81203010: 씬파일러 IV=0.4283, 일반군 IV=0.4711


**결과 종합**

| 변수 | 씬파일러 IV | 일반군 IV | 패턴 |
|---|---|---|---|
| AL012G005 (자택주소 이력) | 0.4056 | 0.1758 | 씬파일러가 더 강함 |
| AL012G011 (직장명 이력) | 0.1722 | 0.0320 | 씬파일러가 더 강함 |
| AL012G019 (휴대폰번호 이력) | 1.3319 | 0.6542 | 씬파일러가 더 강함 (양쪽 다 매우 강함) |
| U81203010 (총자산 구간) | 0.4283 | 0.4711 | **일반군이 오히려 더 강함** |

**생활이력 계열(3개)은 전부 씬파일러에서 예측력이 더 크게 나오고, 자산 계열은
반대로 일반군에서 더 강하다.** 이건 자연스러운 결과다 — 자산은 대출 심사·규모와
직접 연결되니 이미 신용거래 중인 사람들(일반군)한테서 더 잘 맞고, 생활패턴 정보는
금융이력이 없는 사람들의 정보 공백을 메워주는 역할을 하기 때문. 어느 쪽이든
"방향이 뒤집히거나 사라지는" 경우는 없었다 — 즉 노출편향으로 결과가 왜곡된
징후는 발견되지 않았다.

## 8. 결론

**1) PERF3 기반 노출 조건화는 최종 폐기**
- 필터링된 파일 안에서 완화 시도 3종은 전부 무효(원표본 75명 그대로)
- 원본 데이터로 서로 다른 두 가지 방법을 다시 시도함:
  - 패널 기반 탈출자 추적 → 노출 51,917명 확보, 그중 PERF3=1은 2,348명, 그 안 실제 연체(PERF4)는 1명
  - 정의 완화(0건→1건 이하, 2022년 기준) → 노출(PERF3=1) 2,191명 확보(41배 증가), 그 안 실제 연체(PERF4)는 0명
- 두 방법 모두 노출 표본 확보에는 성공했지만, "노출 직후 단기간 내 연체"라는
  마지막 단계에서 동일하게 표본이 소진됨 → 이는 방법의 한계가 아니라
  이 사건 자체가 원래 희귀한 현상이라는 데이터 구조적 특성으로 판단됨

**2) 최종 타겟변수는 PERF2로 채택 권장**
- PERF1(90일↑연체)·PERF4(60일↑연체, 6개월)를 사실상 포함하는 가장 넓은 기준
- 표본 4,668명으로 4개 라벨 중 가장 크고, 통합타겟(PERF_any, 4,679명)과 거의 동일

**3) "노출편향으로 결과가 왜곡될 것"이라는 우려는 검증 결과 실제로 일어나지 않음**
- 노출 문제가 없는 일반 고객군과 씬파일러군에서 대안변수-연체 관계를 비교
- 4개 변수 모두 방향이 일치, IV 격차는 불균형 때문(착시)이 아니라 리샘플링
  검증으로 실제 차이임을 확인
- 오히려 생활이력 계열 변수는 씬파일러에서 예측력이 더 크게 나타나, "대안정보가
  전통정보 공백을 메운다"는 프로젝트 핵심 가설을 뒷받침하는 근거가 됨

**4) 남은 과제**
- 이 검증은 Track A, 2022년 스냅샷 기준. 다른 연도에서도 같은 패턴이
  재현되는지 확인하면 근거가 더 탄탄해짐
- PERF2 채택 시 "노출 여부와 무관하게 실제 위험을 학습하고 있는지"를 추후
  SHAP 결과로 한 번 더 사후 점검하는 것을 권장
